# WaveForge — Brain Haemorrhage Dataset Generator

**Just click Run All — fully automated.**

### Strategy
- **Phase 1 (Cell 4):** Both GPUs generate training samples in parallel → 800 samples each → **1600 total in ~4.5h**
- **Phase 2 (Cell 5):** Both GPUs generate test samples in parallel → 200 each → **400 total in ~1.1h**
- Total wall time: **~5.5h** on 2×T4

| Property | Value |
|----------|-------|
| Frequency | 1.0 GHz | Grid | 64³ at 3mm/cell |
| Antennas | 8-element ring | Steps | 300 per TX |
| Classes | 0=healthy 1=epidural 2=subdural 3=intracerebral |
| Train | 1600 samples — both GPUs, seeds 0–15999 |
| Test | 400 samples — both GPUs, seeds 10M–10M+3999 |

**Accelerator:** GPU T4 x2

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib, threading, time, json, datetime

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np
assert torch.cuda.is_available(), 'No GPU — enable T4 x2 accelerator!'
N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name} {p.total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}\n✅ Ready')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configuration                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
FREQ_GHZ    = 1.0
GRID_SIZE   = 64
DX_MM       = 3.0
N_TX        = 8
RING_RADIUS = 30
N_STEPS     = 300

# Both GPUs generate training samples → split evenly
N_TRAIN_TOTAL = 1600
N_TRAIN_PER_GPU = N_TRAIN_TOTAL // 2   # 800 each

# Both GPUs generate test samples → split evenly
N_TEST_TOTAL = 400
N_TEST_PER_GPU = N_TEST_TOTAL // 2     # 200 each

OUTPUT_ROOT  = pathlib.Path('/kaggle/working/brain_haemorrhage_dataset')
TRAIN_DIR_G0 = OUTPUT_ROOT / 'train_gpu0'
TRAIN_DIR_G1 = OUTPUT_ROOT / 'train_gpu1'
TEST_DIR_G0  = OUTPUT_ROOT / 'test_gpu0'
TEST_DIR_G1  = OUTPUT_ROOT / 'test_gpu1'
TRAIN_DIR    = OUTPUT_ROOT / 'train'   # merged after both GPUs done
TEST_DIR     = OUTPUT_ROOT / 'test'

GPU0 = 'cuda:0'
GPU1 = 'cuda:1' if N_GPUS > 1 else 'cuda:0'

# Separate seed ranges per GPU — no sample overlap
TRAIN_SEED_G0 = 0
TRAIN_SEED_G1 = 100_000       # GPU1 train seeds
TEST_SEED_G0  = 10_000_000    # test seeds — separate space from train
TEST_SEED_G1  = 10_100_000

sec_per = N_TX * N_STEPS * GRID_SIZE**3 / 62e6 * 2  # ~62 Mc/s per T4
print(f'~{sec_per:.0f}s per sample')
print(f'Phase 1 (train): {N_TRAIN_PER_GPU} samples × 2 GPUs → ~{sec_per*N_TRAIN_PER_GPU/3600:.1f}h')
print(f'Phase 2 (test):  {N_TEST_PER_GPU} samples × 2 GPUs → ~{sec_per*N_TEST_PER_GPU/3600:.1f}h')
print(f'Total: ~{sec_per*(N_TRAIN_PER_GPU+N_TEST_PER_GPU)/3600:.1f}h')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Validate 4 samples before full run                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from datasets.generator import BrainDatasetGenerator

print('Validation: 4 samples (one per class)...')
val_gen = BrainDatasetGenerator(
    output_dir=str(OUTPUT_ROOT / 'validation'),
    freq_hz=FREQ_GHZ * 1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
    n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
    device=GPU0,
)
val_manifest = val_gen.generate_balanced_dataset(
    n_samples=4, phantom_id='train', base_seed=42, show_progress=True
)
if val_manifest['n_completed'] < 2:
    raise RuntimeError('Validation failed.')
s = np.load(val_manifest['sample_paths'][0], allow_pickle=True)
print(f'signals_scattered: {s["signals_scattered"].shape}  label={s["label"]}  energy={( s["signals_scattered"]**2).sum():.2e}')
print('✅ Validation passed')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — PHASE 1: Both GPUs generate TRAINING samples in parallel     ║
# ║  GPU0: samples 0–799  |  GPU1: samples 800–1599                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
results = {}; errors = {}

def run_train_gpu(gpu, out_dir, seed, n):
    key = gpu
    try:
        print(f'[{gpu}] Starting {n} train samples...')
        gen = BrainDatasetGenerator(
            output_dir=str(out_dir),
            freq_hz=FREQ_GHZ*1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        results[key] = gen.generate_balanced_dataset(
            n_samples=n, phantom_id='train', base_seed=seed, show_progress=True,
        )
        print(f'[{gpu}] Done: {results[key]["n_completed"]} samples')
    except Exception as e:
        errors[key] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_train_gpu, args=(GPU0, TRAIN_DIR_G0, TRAIN_SEED_G0, N_TRAIN_PER_GPU)),
    threading.Thread(target=run_train_gpu, args=(GPU1, TRAIN_DIR_G1, TRAIN_SEED_G1, N_TRAIN_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()

if errors: raise RuntimeError(f'Train errors: {errors}')

# Merge GPU0 + GPU1 results into single train manifest
import shutil
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
train_paths, train_labels, train_types, train_ages = [], [], [], []
train_class_counts = {0:0, 1:0, 2:0, 3:0}

for gpu_key, src_dir in [('cuda:0', TRAIN_DIR_G0), ('cuda:1', TRAIN_DIR_G1)]:
    m = results[gpu_key]
    for old_path, lbl, bt, ba in zip(m['sample_paths'], m['labels'], m['bleed_types'], m['bleed_ages']):
        new_idx = len(train_paths)
        new_path = TRAIN_DIR / f'sample_{new_idx:06d}.npz'
        shutil.copy(old_path, new_path)
        train_paths.append(str(new_path))
        train_labels.append(lbl)
        train_types.append(bt)
        train_ages.append(ba)
        train_class_counts[lbl] += 1

train_manifest = {
    'n_completed': len(train_paths), 'n_failed': 0,
    'class_counts': train_class_counts,
    'sample_paths': train_paths, 'labels': train_labels,
    'bleed_types': train_types, 'bleed_ages': train_ages,
}
print(f'\n✅ Phase 1 done in {(time.time()-t0)/3600:.2f}h  |  {len(train_paths)} train samples')
print(f'   Class counts: {train_class_counts}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — PHASE 2: Both GPUs generate TEST samples in parallel         ║
# ║  Seeds from 10M space — guaranteed no overlap with training             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
test_results = {}; test_errors = {}

def run_test_gpu(gpu, out_dir, seed, n):
    key = gpu
    try:
        print(f'[{gpu}] Starting {n} test samples...')
        gen = BrainDatasetGenerator(
            output_dir=str(out_dir),
            freq_hz=FREQ_GHZ*1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        test_results[key] = gen.generate_balanced_dataset(
            n_samples=n, phantom_id='test', base_seed=seed, show_progress=True,
        )
        print(f'[{gpu}] Done: {test_results[key]["n_completed"]} test samples')
    except Exception as e:
        test_errors[key] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_test_gpu, args=(GPU0, TEST_DIR_G0, TEST_SEED_G0, N_TEST_PER_GPU)),
    threading.Thread(target=run_test_gpu, args=(GPU1, TEST_DIR_G1, TEST_SEED_G1, N_TEST_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()

if test_errors: raise RuntimeError(f'Test errors: {test_errors}')

# Merge
TEST_DIR.mkdir(parents=True, exist_ok=True)
test_paths, test_labels, test_types, test_ages = [], [], [], []
test_class_counts = {0:0, 1:0, 2:0, 3:0}

for gpu_key, src_dir in [('cuda:0', TEST_DIR_G0), ('cuda:1', TEST_DIR_G1)]:
    m = test_results[gpu_key]
    for old_path, lbl, bt, ba in zip(m['sample_paths'], m['labels'], m['bleed_types'], m['bleed_ages']):
        new_idx = len(test_paths)
        new_path = TEST_DIR / f'sample_{new_idx:06d}.npz'
        shutil.copy(old_path, new_path)
        test_paths.append(str(new_path))
        test_labels.append(lbl)
        test_types.append(bt)
        test_ages.append(ba)
        test_class_counts[lbl] += 1

test_manifest = {
    'n_completed': len(test_paths), 'class_counts': test_class_counts,
    'sample_paths': test_paths, 'labels': test_labels,
    'bleed_types': test_types, 'bleed_ages': test_ages,
}
print(f'\n✅ Phase 2 done in {(time.time()-t0)/3600:.2f}h  |  {len(test_paths)} test samples')
print(f'   Class counts: {test_class_counts}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Save master manifest                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
try:
    commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
except: commit = 'unknown'

master = {
    'version': '1.2', 'created_at': datetime.datetime.now().isoformat(),
    'waveforge_commit': commit,
    'n_total': len(train_paths) + len(test_paths),
    'n_train': len(train_paths), 'n_test': len(test_paths),
    'train_class_counts': train_class_counts,
    'test_class_counts':  test_class_counts,
    'class_names': {0:'healthy',1:'epidural',2:'subdural',3:'intracerebral'},
    'phantom_design': 'unique_per_sample',
    'freq_hz': FREQ_GHZ*1e9, 'grid_shape': [GRID_SIZE]*3,
    'dx_mm': DX_MM, 'n_tx': N_TX, 'n_rx': N_TX, 'n_steps': N_STEPS,
    'train_dir': str(TRAIN_DIR), 'test_dir': str(TEST_DIR),
    'gpu_strategy': 'both_gpus_train_then_test',
    'seed_ranges': {
        'train_gpu0': f'{TRAIN_SEED_G0}–{TRAIN_SEED_G0+N_TRAIN_PER_GPU*10}',
        'train_gpu1': f'{TRAIN_SEED_G1}–{TRAIN_SEED_G1+N_TRAIN_PER_GPU*10}',
        'test_gpu0':  f'{TEST_SEED_G0}–{TEST_SEED_G0+N_TEST_PER_GPU*10}',
        'test_gpu1':  f'{TEST_SEED_G1}–{TEST_SEED_G1+N_TEST_PER_GPU*10}',
    },
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'dataset_manifest.json'
with open(manifest_path, 'w') as f: json.dump(master, f, indent=2)
print(f'Total: {master["n_total"]} samples  |  commit: {commit}')
print(f'Manifest: {manifest_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Visualise one sample per class                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
label_to_name = {0:'Healthy',1:'Epidural',2:'Subdural',3:'Intracerebral'}
samples_by_label = {}
for path, label in zip(train_manifest['sample_paths'], train_manifest['labels']):
    if label not in samples_by_label:
        samples_by_label[label] = np.load(path, allow_pickle=True)
    if len(samples_by_label) == 4: break

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle('WaveForge Brain Dataset — One Sample per Class', fontsize=13, fontweight='bold')
for row, label in enumerate(sorted(samples_by_label)):
    s = samples_by_label[label]
    name = label_to_name[label]
    age  = str(s['bleed_age']); r_mm = float(s['bleed_radius_mm'])
    scat = s['signals_scattered']
    t_ns = np.arange(scat.shape[2]) * float(s['dt_s']) * 1e9
    ax = axes[row,0]
    for rx in range(scat.shape[1]): ax.plot(t_ns, scat[0,rx], alpha=0.6, lw=0.8)
    ax.set(title=name if age=='none' else f'{name} ({age}, r={r_mm:.0f}mm)',
           xlabel='Time (ns)', ylabel='Scattered Ez (V/m)')
    ax.grid(alpha=0.3)
    ax2 = axes[row,1]
    im = ax2.imshow(s['das_image'], cmap='hot', origin='lower',
                    extent=[0,GRID_SIZE*DX_MM,0,GRID_SIZE*DX_MM])
    ax2.set(title='DAS backprojection', xlabel='x (mm)', ylabel='y (mm)')
    if r_mm > 0:
        ax2.plot(float(s['bleed_center_mm'][0]), float(s['bleed_center_mm'][1]),
                 'c+', markersize=15, mew=2, label='true bleed')
        ax2.legend(fontsize=8)
    plt.colorbar(im, ax=ax2)
plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
plt.savefig('docs/assets/brain_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Package for download                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil
OUT = pathlib.Path('/kaggle/working/waveforge_brain_outputs')
OUT.mkdir(exist_ok=True)
shutil.copy(manifest_path, OUT)
vis = pathlib.Path('docs/assets/brain_dataset_samples.png')
if vis.exists(): shutil.copy(vis, OUT)
train_files = sorted(TRAIN_DIR.glob('*.npz'))
test_files  = sorted(TEST_DIR.glob('*.npz'))
total_mb = sum(f.stat().st_size for f in train_files+test_files)/1e6
print(f'Train: {len(train_files)} | Test: {len(test_files)} | Size: {total_mb:.1f} MB')
print('🏁 Dataset generation complete!')